Import ``maths``, ``MultiVisit``, ``ufloat``, and ``emcee``:

In [ ]:
from uncertainties.umath import cos, radians, sin, sqrt
from pycheops import MultiVisit
from uncertainties import ufloat
import emcee

Pick up ``.dataset`` :

In [ ]:
# Create variables with target and tag name so that we can create a unique HDF5 backend file name below
target = 'WASP-18'
tag = 'glint_scale'
new_tag = 'analysis'     # Name of MultiVisit file to save after analysis is complete

Define ``MultiVisit`` parameters:

In [ ]:
from pathlib import Path
import requests
from pycheops.core import load_config

# Refresh pycheops' catalog cache from the public GitHub repository.
csv_url = "https://raw.githubusercontent.com/iastro-pt/SWEET-Cat/master/SWEETCat_topcat.csv"
config = load_config()
cache_dir = Path(config["DEFAULT"]["data_cache_path"])
cache_dir.mkdir(parents=True, exist_ok=True)

response = requests.get(csv_url, timeout=30)
response.raise_for_status()
(cache_dir / "sweetcat.csv").write_bytes(response.content)

data_dir = Path("data")
M = MultiVisit(
    target=target,
    datadir=data_dir,
    tag=tag,
    id_kws={"dace": False, "match_arcsec": None})

Set known priors here:

In [ ]:
from math import pi
Tc = ufloat(2458501.324483, 0.000019)
P = ufloat(0.94145299, 0.00000087)
aR = ufloat(3.48, 0.17)     # a/R_*
k = ufloat(0.1018, 0.0011)
i = ufloat(86.0, 2.5)
e = float(0)
om_deg = float(0)     # Ignored actual om = (-96 +/- 10) value

Values calculated from priors:

In [ ]:
b = aR * cos(radians(i))
D = k**2
W = (1 / aR) * sqrt((1 + k)**2 - b**2) / pi
T_0 = Tc - 2457000     # Kokori et al. 2023 BJD_TDB-2457000
om_rad = radians(om_deg)
f_c = sqrt(e)*cos(om_rad)
f_s = sqrt(e)*sin(om_rad)

RUN BURN-IN PHASE:

In [ ]:
nwalkers=64     # Set chosen number of walkers for the MCMC sampler; same for all runs

result0 = M.fit_transit(
    D=[D.n/2,D.n,D.n*2],
    W=[W.n/2,W.n,W.n*2],
    b=[0,b.n,1],
    T_0=[T_0.n-0.01,T_0.n+0.01],
    P=[P.n-0.0001,P.n+0.0001],
    f_c=f_c,
    f_s=f_s,
    unwrap=True,
    unroll=True,
    nroll=3,
    h_2=[0,0.4,1],
    nwalkers=nwalkers,
    burn=128,
    steps=128,
    progress=False)
ndim = M.__sampler__.ndim
print(f'No. of free parameters {ndim = }')
nwalkers = M.__sampler__.nwalkers
print(f'No. of walkers {nwalkers = }')

# Set up the backend
# Don't forget to clear it in case the file already exists
filename = f"{target}_{new_tag}.h5"
print(f'Backend being saved to file {filename}')
backend = emcee.backends.HDFBackend(filename)
backend.reset(nwalkers, ndim)
result1 = M.fit_transit(
    D=[D.n/2,D.n,D.n*2],
    W=[W.n/2,W.n,W.n*2],
    b=[0,b.n,1],
    T_0=[T_0.n-0.01,T_0.n+0.01],
    P=[P.n-0.0001,P.n+0.0001],
    f_c=f_c,
    f_s=f_s,
    unwrap=True,
    unroll=True,
    nroll=3,
    h_2=[0,0.4,1],
    nwalkers=nwalkers,
    burn=256,
    steps=512,
    backend=backend)

Save ``.multivisit`` file:

In [ ]:
M.save(tag=new_tag, overwrite = True)

MAIN RUN:

In [ ]:
new_backend = emcee.backends.HDFBackend(filename)
print("Initial size: {0}".format(new_backend.iteration))
result1 = M.fit_transit(
    D=[D.n/2,D.n,D.n*2],
    W=[W.n/2,W.n,W.n*2],
    b=[0,b.n,1],
    T_0=[T_0.n-0.01,T_0.n+0.01],
    P=[P.n-0.0001,P.n+0.0001],
    f_c=f_c,
    f_s=f_s,
    unwrap=True,
    unroll=True,
    nroll=3,
    h_2=[0,0.4,1],
    nwalkers=nwalkers,
    steps=1024,
    backend=new_backend)

Plot a trail plot:

In [ ]:
fig = M.trail_plot()
#fig.savefig("trail_plot.pdf", bbox_inches="tight")

Plot a corner plot:

In [ ]:
fig = M.corner_plot()
#fig.savefig("corner_plot.pdf", bbox_inches="tight")

Print results (the option ``min_correl`` restricts the number of parameter correlation values in the report):

In [ ]:
fit_report = M.fit_report(min_correl=0.5)
print(fit_report)

#with open("fit_report.txt", "w", encoding="utf-8") as file:
#    file.write(fit_report)

Option to save ``.multivisit`` file again:

In [ ]:
M.save(tag=new_tag, overwrite = True)